# How many max features?

In [1]:
# !pip install mlflow boto3 awscli

In [2]:
# !aws configure

In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://3.110.185.110:5000/")

d:\MLops\MLops-Youtube-Sentiment-Insights\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Set or create an experiment
mlflow.set_experiment("Experiment 3 - TfIdf Trigram max_features")

2026/06/21 09:02:19 INFO mlflow.tracking.fluent: Experiment with name 'Experiment 3 - TfIdf Trigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://yoosuf520-mlflow-s3/6', creation_time=1782012739635, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1782012739635, lifecycle_stage='active', name='Experiment 3 - TfIdf Trigram max_features', tags={}, trace_location=None, workspace='default'>

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [6]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [7]:
# Step 1: Function to run the experiment
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1, 3)  # Trigram setting

    # Step 2: Vectorization using TF-IDF with varying max_features
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"TFIDF_Trigrams_max_features_{max_features}")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, max_features={max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_{max_features}")

# Step 6: Test various max_features values
max_features_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

for max_features in max_features_values:
    run_experiment_tfidf_max_features(max_features)

2026/06/21 09:04:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 09:07:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmp08_0goiz\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 
2026/06/21 09:07:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run TFIDF_Trigrams_max_features_1000 at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6/runs/5bcac647c9b648d1bddc385fd3f33503
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6


2026/06/21 09:12:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 09:13:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run TFIDF_Trigrams_max_features_2000 at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6/runs/93b3414ac01b469886faa1157429d7a5
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6


2026/06/21 09:18:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 09:19:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

🏃 View run TFIDF_Trigrams_max_features_8000 at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6/runs/dd44410ae3ab47acbec2f144a650927d
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6


2026/06/21 09:42:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 09:43:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run TFIDF_Trigrams_max_features_9000 at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6/runs/cebd6eb62afc4b7db6eb214dd16139af
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6


2026/06/21 09:45:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/21 09:46:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run TFIDF_Trigrams_max_features_10000 at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6/runs/7db05b4e59e6434780545377e0543e48
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/6
